# 🛰️ Complete RSVQA Dataset Creator for Google Colab

This notebook allows you to:
1. Download **cloud-free Sentinel-2 satellite images** from Google Earth Engine
2. Download **OpenStreetMap (OSM) data** for the same areas
3. Create custom **RSVQA datasets** for Visual Question Answering

## 📋 Prerequisites
- Google account (for Colab)
- Google Earth Engine account (free signup at https://earthengine.google.com/)

## 🚀 Quick Start
1. Run all cells in order (Runtime → Run all)
2. Authenticate with Google Earth Engine when prompted
3. Modify the locations in the "Download Data" section
4. Download your results!

---

## 📦 Step 1: Install Required Packages

This will install all necessary dependencies.

In [ ]:
%%capture
# Install packages quietly
!pip install earthengine-api geemap rasterio geopandas osmnx matplotlib requests numpy

print("✓ All packages installed successfully!")

## 🔐 Step 2: Authenticate with Google Earth Engine

**IMPORTANT**: First time users must sign up at https://earthengine.google.com/

Click the link below and authorize access to your Google account.

In [ ]:
import ee

try:
    ee.Initialize()
    print("✓ Already authenticated with Google Earth Engine!")
except:
    print("Authenticating with Google Earth Engine...")
    print("Please follow the link and authorize access.")
    ee.Authenticate()
    ee.Initialize()
    print("✓ Authentication successful!")

## 🔧 Step 3: Define the Downloader Class

This cell contains the complete downloader implementation.

In [ ]:
import ee
import geemap
import requests
import json
import os
from datetime import datetime, timedelta
from pathlib import Path
import time
from typing import Dict, List, Optional
import numpy as np

class RSVQADownloader:
    """Download cloud-free Sentinel-2 images with OSM data for Google Colab."""

    def __init__(self, output_dir: str = "./downloads"):
        """
        Initialize the downloader.

        Args:
            output_dir: Directory to save downloaded files
        """
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # Create subdirectories
        self.images_dir = self.output_dir / "images"
        self.osm_dir = self.output_dir / "osm_data"
        self.metadata_dir = self.output_dir / "metadata"

        for dir_path in [self.images_dir, self.osm_dir, self.metadata_dir]:
            dir_path.mkdir(parents=True, exist_ok=True)

        print("✓ Downloader initialized")
        print(f"  Output directory: {self.output_dir}")

    def get_cloud_free_composite(
        self,
        bbox: List[float],
        start_date: str,
        end_date: str,
        cloud_percentage: int = 10
    ) -> Optional[ee.Image]:
        """
        Get cloud-free Sentinel-2 composite for a bounding box.

        Args:
            bbox: [min_lon, min_lat, max_lon, max_lat]
            start_date: Start date (YYYY-MM-DD)
            end_date: End date (YYYY-MM-DD)
            cloud_percentage: Maximum cloud cover percentage

        Returns:
            Earth Engine Image or None if no suitable images found
        """
        roi = ee.Geometry.Rectangle(bbox)

        # Query Sentinel-2 Surface Reflectance
        collection = (ee.ImageCollection('COPERNICUS/S2_SR')
                     .filterBounds(roi)
                     .filterDate(start_date, end_date)
                     .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_percentage))
                     .select(['B4', 'B3', 'B2']))  # RGB bands

        count = collection.size().getInfo()
        print(f"  Found {count} images with <{cloud_percentage}% cloud cover")

        if count < 3:
            print(f"  ⚠️ Warning: Only {count} images found")
            if count == 0:
                return None

        # Create median composite
        composite = collection.median().clip(roi)
        return composite

    def check_land_coverage(self, image: ee.Image, roi: ee.Geometry) -> Dict:
        """Check if the image contains land (not just ocean)."""
        stats = image.reduceRegion(
            reducer=ee.Reducer.mean().combine(
                ee.Reducer.stdDev(), '', True
            ),
            geometry=roi,
            scale=100,
            maxPixels=1e9
        ).getInfo()

        mean_brightness = np.mean([stats.get('B4_mean', 0),
                                   stats.get('B3_mean', 0),
                                   stats.get('B2_mean', 0)])
        std_dev = np.mean([stats.get('B4_stdDev', 0),
                          stats.get('B3_stdDev', 0),
                          stats.get('B2_stdDev', 0)])

        is_likely_land = mean_brightness > 500 or std_dev > 150

        return {
            'mean_brightness': float(mean_brightness),
            'std_dev': float(std_dev),
            'is_likely_land': is_likely_land
        }

    def fetch_osm_data(self, bbox: List[float], timeout: int = 180) -> Dict:
        """
        Fetch OSM data using Overpass API.

        Args:
            bbox: [min_lon, min_lat, max_lon, max_lat]
            timeout: API timeout in seconds

        Returns:
            GeoJSON FeatureCollection
        """
        features = [
            'building', 'highway', 'landuse', 'natural', 'waterway',
            'amenity', 'leisure', 'railway', 'aeroway'
        ]

        bbox_str = f"{bbox[1]},{bbox[0]},{bbox[3]},{bbox[2]}"  # S,W,N,E

        queries = []
        for feature in features:
            queries.append(f'way["{feature}"]({bbox_str});')
            queries.append(f'relation["{feature}"]({bbox_str});')

        overpass_query = f"""
        [out:json][timeout:{timeout}];
        (
          {''.join(queries)}
        );
        out geom;
        """

        print(f"  Fetching OSM data...")

        try:
            response = requests.post(
                "https://overpass-api.de/api/interpreter",
                data={'data': overpass_query},
                timeout=timeout
            )
            response.raise_for_status()
            osm_data = response.json()
            geojson = self._osm_to_geojson(osm_data, bbox)
            print(f"  ✓ Retrieved {len(geojson['features'])} OSM features")
            return geojson

        except Exception as e:
            print(f"  ⚠️ OSM error: {e}")
            return {"type": "FeatureCollection", "features": [], "error": str(e)}

    def _osm_to_geojson(self, osm_data: Dict, bbox: List[float]) -> Dict:
        """Convert OSM JSON to GeoJSON."""
        features = []

        for element in osm_data.get('elements', []):
            try:
                geometry = None
                if element['type'] == 'way' and 'geometry' in element:
                    coords = [[node['lon'], node['lat']] for node in element['geometry']]
                    if len(coords) > 1:
                        if coords[0] == coords[-1] and len(coords) > 3:
                            geometry = {"type": "Polygon", "coordinates": [coords]}
                        else:
                            geometry = {"type": "LineString", "coordinates": coords}

                elif element['type'] == 'node':
                    geometry = {
                        "type": "Point",
                        "coordinates": [element['lon'], element['lat']]
                    }

                if geometry:
                    properties = element.get('tags', {})
                    properties['osm_id'] = element['id']
                    properties['osm_type'] = element['type']

                    features.append({
                        "type": "Feature",
                        "geometry": geometry,
                        "properties": properties
                    })
            except:
                continue

        return {
            "type": "FeatureCollection",
            "bbox": bbox,
            "features": features,
            "metadata": {
                "source": "OpenStreetMap",
                "attribution": "© OpenStreetMap contributors",
                "retrieved": datetime.now().isoformat()
            }
        }

    def download_image(self, image: ee.Image, bbox: List[float],
                      filename: str, scale: int = 10) -> str:
        """Download image to local filesystem."""
        output_path = self.images_dir / f"{filename}.tif"
        print(f"  Downloading image...")

        geemap.ee_export_image(
            image,
            filename=str(output_path),
            scale=scale,
            region=ee.Geometry.Rectangle(bbox),
            file_per_band=False,
            crs='EPSG:3857'
        )

        print(f"  ✓ Saved: {output_path}")
        return str(output_path)

    def process_location(
        self,
        name: str,
        bbox: List[float],
        cloud_percentage: int = 10,
        scale: int = 10
    ) -> Dict:
        """
        Download Sentinel-2 image + OSM data for a location.

        Args:
            name: Location name
            bbox: [min_lon, min_lat, max_lon, max_lat]
            cloud_percentage: Max cloud cover %
            scale: Resolution in meters

        Returns:
            Dictionary with results
        """
        print(f"\n{'='*60}")
        print(f"📍 Processing: {name}")
        print(f"📦 Bbox: {bbox}")
        print(f"{'='*60}")

        end_date = datetime.now().strftime('%Y-%m-%d')
        start_date = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')

        result = {
            'name': name,
            'bbox': bbox,
            'status': 'failed'
        }

        try:
            # Get cloud-free composite
            print("\n[1/4] 🛰️  Fetching Sentinel-2 composite...")
            image = self.get_cloud_free_composite(
                bbox, start_date, end_date, cloud_percentage
            )

            if image is None:
                result['error'] = "No suitable images found"
                print("  ❌ No images available")
                return result

            # Check land coverage
            print("\n[2/4] 🗺️  Checking land coverage...")
            roi = ee.Geometry.Rectangle(bbox)
            land_check = self.check_land_coverage(image, roi)
            print(f"  Brightness: {land_check['mean_brightness']:.1f}")
            print(f"  Variation: {land_check['std_dev']:.1f}")

            if not land_check['is_likely_land']:
                print("  ⚠️ May be mostly ocean")

            # Download image
            print("\n[3/4] 💾 Downloading image...")
            image_path = self.download_image(image, bbox, name, scale)
            result['image_path'] = image_path

            # Fetch OSM data
            print("\n[4/4] 🗺️  Fetching OSM data...")
            osm_data = self.fetch_osm_data(bbox)

            osm_path = self.osm_dir / f"{name}_osm.geojson"
            with open(osm_path, 'w') as f:
                json.dump(osm_data, f, indent=2)
            result['osm_path'] = str(osm_path)

            # Save metadata
            metadata = {
                'name': name,
                'bbox': bbox,
                'date_range': {'start': start_date, 'end': end_date},
                'cloud_percentage': cloud_percentage,
                'scale_meters': scale,
                'land_coverage': land_check,
                'osm_features': len(osm_data['features']),
                'processed_at': datetime.now().isoformat()
            }

            metadata_path = self.metadata_dir / f"{name}_metadata.json"
            with open(metadata_path, 'w') as f:
                json.dump(metadata, f, indent=2)
            result['metadata_path'] = str(metadata_path)

            result['status'] = 'success'
            result['metadata'] = metadata
            print(f"\n✅ Successfully processed {name}!")

        except Exception as e:
            print(f"\n❌ Error: {e}")
            result['error'] = str(e)

        return result

print("✓ RSVQADownloader class defined successfully!")

## 🎯 Step 4: Initialize the Downloader

Create an instance of the downloader.

In [ ]:
# Initialize downloader
downloader = RSVQADownloader(output_dir="./rsvqa_downloads")

print("\n🚀 Ready to download satellite data!")

## 📍 Example 1: Download a Single Location

Let's download Manhattan, NYC as a test.

In [ ]:
# Download Manhattan
result = downloader.process_location(
    name='manhattan_test',
    bbox=[-74.02, 40.75, -73.97, 40.80],  # [min_lon, min_lat, max_lon, max_lat]
    cloud_percentage=10,
    scale=10  # 10m resolution
)

# Display result
print("\n" + "="*60)
print("RESULT SUMMARY")
print("="*60)
print(json.dumps(result, indent=2, default=str))

## 🌍 Example 2: Download Multiple Diverse Locations

Download several locations representing different landscapes.

In [ ]:
# Define locations
locations = [
    {
        'name': 'manhattan_nyc',
        'bbox': [-74.02, 40.75, -73.97, 40.80],
        'cloud_percentage': 10,
        'description': 'Urban - Manhattan'
    },
    {
        'name': 'iowa_farmland',
        'bbox': [-93.65, 41.55, -93.55, 41.65],
        'cloud_percentage': 10,
        'description': 'Agricultural - Iowa'
    },
    {
        'name': 'arizona_desert',
        'bbox': [-111.75, 33.45, -111.65, 33.55],
        'cloud_percentage': 5,
        'description': 'Desert - Arizona'
    },
    {
        'name': 'san_francisco_bay',
        'bbox': [-122.45, 37.75, -122.35, 37.85],
        'cloud_percentage': 10,
        'description': 'Coastal - San Francisco'
    },
    {
        'name': 'amazon_rainforest',
        'bbox': [-60.1, -3.1, -60.0, -3.0],
        'cloud_percentage': 20,
        'description': 'Rainforest - Amazon'
    }
]

# Download all locations
results = []
for loc in locations:
    print(f"\n🌍 Starting: {loc['description']}")
    
    result = downloader.process_location(
        name=loc['name'],
        bbox=loc['bbox'],
        cloud_percentage=loc['cloud_percentage']
    )
    
    results.append(result)
    
    # Rate limiting
    time.sleep(3)

# Summary
print("\n" + "="*60)
print("📊 BATCH DOWNLOAD SUMMARY")
print("="*60)
successful = [r for r in results if r['status'] == 'success']
failed = [r for r in results if r['status'] == 'failed']

print(f"Total: {len(results)}")
print(f"✅ Successful: {len(successful)}")
print(f"❌ Failed: {len(failed)}")

if successful:
    print("\n✅ Successfully downloaded:")
    for r in successful:
        osm_count = r.get('metadata', {}).get('osm_features', 0)
        print(f"  • {r['name']}: {osm_count} OSM features")

if failed:
    print("\n❌ Failed:")
    for r in failed:
        print(f"  • {r['name']}: {r.get('error', 'Unknown')}")

## 🔍 Step 5: Verify Downloaded Files

In [ ]:
# List downloaded files
!echo "📁 Downloaded Images:"
!ls -lh rsvqa_downloads/images/

!echo "\n📁 OSM Data Files:"
!ls -lh rsvqa_downloads/osm_data/

!echo "\n📁 Metadata Files:"
!ls -lh rsvqa_downloads/metadata/

## 📊 Step 6: Visualize the Data

Let's visualize one of the downloaded satellite images.

In [ ]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Find the first available image
image_files = list(Path('rsvqa_downloads/images/').glob('*.tif'))

if image_files:
    image_path = image_files[0]
    print(f"Visualizing: {image_path.name}\n")
    
    # Open and read the image
    with rasterio.open(image_path) as src:
        # Read RGB bands
        rgb = src.read([1, 2, 3])
        
        # Transpose to (height, width, channels)
        rgb = np.transpose(rgb, (1, 2, 0))
        
        # Normalize for display (adjust based on data range)
        rgb_norm = np.clip(rgb / 3000 * 255, 0, 255).astype(np.uint8)
        
        # Create figure
        fig, ax = plt.subplots(figsize=(14, 14))
        ax.imshow(rgb_norm)
        ax.set_title(f'Cloud-Free Sentinel-2 Image: {image_path.stem}', 
                     fontsize=16, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        
        # Print metadata
        print(f"Image properties:")
        print(f"  Shape: {rgb.shape}")
        print(f"  CRS: {src.crs}")
        print(f"  Bounds: {src.bounds}")
else:
    print("❌ No images found. Run the download cells first!")

## 🗺️ Step 7: Analyze OSM Data

Examine the OpenStreetMap features we downloaded.

In [ ]:
import json
from collections import Counter
from pathlib import Path

# Find first OSM file
osm_files = list(Path('rsvqa_downloads/osm_data/').glob('*.geojson'))

if osm_files:
    osm_path = osm_files[0]
    print(f"Analyzing: {osm_path.name}\n")
    
    with open(osm_path, 'r') as f:
        osm_data = json.load(f)
    
    features = osm_data['features']
    print(f"📊 Total OSM features: {len(features)}\n")
    
    # Count feature types
    feature_types = []
    for feature in features:
        props = feature['properties']
        for key in ['building', 'highway', 'landuse', 'natural', 
                    'amenity', 'waterway', 'leisure', 'railway']:
            if key in props:
                feature_types.append(f"{key}={props[key]}")
                break
    
    counts = Counter(feature_types)
    
    print("🏆 Top 15 feature types:")
    for i, (feature, count) in enumerate(counts.most_common(15), 1):
        print(f"  {i:2d}. {feature:30s} : {count:5d}")
    
    # Geometry type distribution
    geom_types = Counter([f['geometry']['type'] for f in features])
    print("\n📐 Geometry types:")
    for geom_type, count in geom_types.items():
        print(f"  • {geom_type}: {count}")
        
else:
    print("❌ No OSM data found. Run the download cells first!")

## 🎯 Step 8: Download Your Own Custom Location

Use this cell to download any location you want!

**Finding coordinates:**
- BoundingBox Tool: https://boundingbox.klokantech.com/ (select CSV format)
- Google Maps: Right-click → "What's here?"
- GeoJSON.io: https://geojson.io/

In [ ]:
# YOUR CUSTOM LOCATION
# Replace these values with your area of interest

my_location = {
    'name': 'my_study_area',  # Change this
    'bbox': [-122.5, 37.7, -122.4, 37.8],  # Change this: [min_lon, min_lat, max_lon, max_lat]
    'cloud_percentage': 10,  # Adjust if needed (5-20)
    'scale': 10  # Resolution in meters (10 = 10m/pixel)
}

print(f"🎯 Downloading custom location: {my_location['name']}")
print(f"📍 Coordinates: {my_location['bbox']}\n")

result = downloader.process_location(
    name=my_location['name'],
    bbox=my_location['bbox'],
    cloud_percentage=my_location['cloud_percentage'],
    scale=my_location['scale']
)

print("\n" + "="*60)
if result['status'] == 'success':
    print("✅ SUCCESS!")
    print(f"Image: {result['image_path']}")
    print(f"OSM data: {result['osm_path']}")
else:
    print("❌ FAILED")
    print(f"Error: {result.get('error', 'Unknown error')}")

## 📋 Step 9: View Metadata

Examine the metadata for one of the downloaded locations.

In [ ]:
import json
from pathlib import Path

# Find first metadata file
metadata_files = list(Path('rsvqa_downloads/metadata/').glob('*.json'))

if metadata_files:
    metadata_path = metadata_files[0]
    print(f"📄 Metadata: {metadata_path.name}\n")
    
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    
    print(json.dumps(metadata, indent=2))
else:
    print("❌ No metadata found. Run the download cells first!")

## 💾 Step 10: Download Results to Your Computer

Package everything into a ZIP file and download it.

In [ ]:
from google.colab import files
import os

# Create ZIP archive
print("📦 Creating ZIP archive...")
!zip -r rsvqa_dataset.zip rsvqa_downloads/ -q

# Get file size
size_mb = os.path.getsize('rsvqa_dataset.zip') / (1024 * 1024)
print(f"✓ Archive created: {size_mb:.2f} MB")

# Download
print("\n⬇️ Downloading to your computer...")
files.download('rsvqa_dataset.zip')

print("✅ Done! Check your browser's download folder.")

## ☁️ Optional: Save to Google Drive

Save your downloads to Google Drive so they persist between sessions.

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Copy downloads to Drive
!mkdir -p /content/drive/MyDrive/RSVQA_Data
!cp -r rsvqa_downloads/* /content/drive/MyDrive/RSVQA_Data/

print("✅ Data saved to Google Drive: MyDrive/RSVQA_Data/")

## 🎓 Next Steps

Now that you have satellite images and OSM data, you can:

1. **Tile the images** into smaller patches (256×256 pixels)
2. **Generate VQA questions** from the OSM features
3. **Create train/val/test splits**
4. **Train a VQA model**

### Quick Commands for Processing:

```python
# Tile images into patches
!python prepare_sentinel2_data.py \
    --input_dir rsvqa_downloads/images \
    --output_dir dataset/tiles \
    --patch_size 256

# Create dataset splits
!python create_dataset_splits.py \
    --data_dir dataset/tiles \
    --output_dir dataset/splits
```

---

## 📚 Resources

- **Find coordinates**: https://boundingbox.klokantech.com/
- **Google Earth Engine**: https://earthengine.google.com/
- **OpenStreetMap**: https://www.openstreetmap.org/
- **Sentinel-2 Info**: https://sentinel.esa.int/web/sentinel/missions/sentinel-2

---

## ❓ Troubleshooting

### No images found
- Increase `cloud_percentage` (try 15-20)
- Check if area has Sentinel-2 coverage
- Verify bbox coordinates are correct

### OSM API timeout
- Reduce bounding box size
- Wait a few minutes and retry
- OSM API has rate limits

### Out of memory
- Use smaller bounding boxes
- Reduce number of locations
- Increase scale (e.g., 20m instead of 10m)

### Authentication errors
```python
# Re-authenticate
import ee
ee.Authenticate(force=True)
ee.Initialize()
```

---

## 🎉 You're Done!

You now have a complete pipeline for downloading satellite imagery and OSM data for RSVQA dataset creation.

**Happy coding!** 🚀🛰️